# 🔥 Train YOLO fire detection for the robot (Raspberry Pi 5)

**EN:** Run the cells from top to bottom in **Google Colab** with a GPU. Result: `fire_ncnn_model.zip` → unzip into `server/models/` on the Pi.

**TH:** รันทีละช่องจากบนลงล่างใน **Google Colab** (ต้องเปิด GPU ก่อน) ผลลัพธ์คือไฟล์ `fire_ncnn_model.zip` → นำไปแตกไฟล์ไว้ที่ `server/models/` บน Pi

**Before you start / ก่อนเริ่ม:** `Runtime → Change runtime type → T4 GPU`

| Step | What |
|---|---|
| 1 | Setup + save to Google Drive (so training survives a disconnect) |
| 2 | Download **D-Fire** dataset (~21k images, classes: smoke, fire) |
| 3 | *(optional)* Add your own Pi camera photos from Roboflow |
| 4 | Train YOLO26n |
| 5 | Check accuracy + test images |
| 6 | Export to NCNN (fast on Pi 5) and download |

## 1. Setup

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics kagglehub
import ultralytics; ultralytics.checks()

In [ ]:
# Save training results to Google Drive. Free Colab can disconnect after a few hours;
# with Drive you can continue with the "resume" cell in step 4.
# บันทึกผลเทรนลง Google Drive ถ้า Colab หลุด จะเทรนต่อได้
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/fire_robot_yolo'
import os; os.makedirs(PROJECT, exist_ok=True)

## 2. Download D-Fire dataset

D-Fire: https://github.com/gaiasd/DFireDataset (YOLO-format labels). We download a Kaggle copy of it with `kagglehub`.

If the download fails: download the dataset zip yourself (from the GitHub link above or Kaggle), upload it to Colab, unzip it to `/content/dfire`, and set `DATA_ROOT = '/content/dfire'`.

In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
print(DATA_ROOT)
!find "$DATA_ROOT" -maxdepth 3 -type d | head -30
!find "$DATA_ROOT" -name "*.yaml" | head

In [ ]:
# Find train / val / test image folders automatically and write our own data.yaml.
# หาโฟลเดอร์ train / val / test อัตโนมัติ แล้วสร้าง data.yaml
import glob, yaml, os
from collections import Counter

def find_split(root, names):
    for n in names:
        hits = [d for d in glob.glob(f'{root}/**/{n}/images', recursive=True)] + \
               [d for d in glob.glob(f'{root}/**/images/{n}', recursive=True)]
        if hits:
            return sorted(hits, key=len)[0]
    return None

train_dir = find_split(DATA_ROOT, ['train'])
val_dir   = find_split(DATA_ROOT, ['val', 'valid', 'validation'])
test_dir  = find_split(DATA_ROOT, ['test'])
print('train:', train_dir, '\nval:', val_dir, '\ntest:', test_dir)
assert train_dir, 'train images not found - look at the folder list above and set train_dir by hand'
if val_dir is None:   # no val split: use test as val
    val_dir = test_dir

# Class names. D-Fire uses 0 = smoke, 1 = fire. Use the dataset's own yaml if it has one.
names = ['smoke', 'fire']
for y in glob.glob(f'{DATA_ROOT}/**/*.yaml', recursive=True):
    d = yaml.safe_load(open(y))
    if isinstance(d, dict) and 'names' in d:
        n = d['names']
        names = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
        print('class names from', y)
        break
print('classes:', dict(enumerate(names)))

data = {'train': train_dir, 'val': val_dir, 'names': dict(enumerate(names))}
if test_dir and test_dir != val_dir:
    data['test'] = test_dir
DATA_YAML = '/content/fire.yaml'
yaml.safe_dump(data, open(DATA_YAML, 'w'))
print(open(DATA_YAML).read())

# Sanity check: images and labels per split, boxes per class
def count(img_dir):
    lbl_dir = img_dir.replace('/images', '/labels')
    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    c = Counter()
    for f in glob.glob(f'{lbl_dir}/*.txt'):
        for line in open(f):
            if line.strip():
                c[names[int(line.split()[0])]] += 1
    return len(imgs), dict(c)
for k in ('train', 'val', 'test'):
    if k in data:
        print(k, count(data[k]))

In [ ]:
# Look at a few labelled images to confirm "fire" boxes are really on fire (class ids correct).
# ดูภาพตัวอย่าง: กล่อง "fire" ต้องอยู่บนไฟจริง ถ้าสลับกับ smoke ให้แก้ names ด้านบน
import random, cv2, matplotlib.pyplot as plt
files = random.sample(glob.glob(f'{train_dir}/*.jpg'), 6)
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axs.flat, files):
    im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lf = f.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if os.path.exists(lf):
        for line in open(lf):
            c, x, y, bw, bh = map(float, line.split())
            x1, y1 = int((x - bw / 2) * w), int((y - bh / 2) * h)
            col = (255, 60, 0) if names[int(c)] == 'fire' else (180, 180, 180)
            cv2.rectangle(im, (x1, y1), (x1 + int(bw * w), y1 + int(bh * h)), col, 3)
            cv2.putText(im, names[int(c)], (x1, max(20, y1 - 5)), 0, 0.9, col, 2)
    ax.imshow(im); ax.axis('off')
plt.show()

## 3. *(Optional, recommended)* Add your own photos

A model works best on photos from **your own Pi camera** in the place the robot will run.
Include **"not fire" photos** too (orange lights, sunset, red objects) with **no boxes**. These stop false alarms.

1. Take 200–500 photos with the Pi camera (`rpicam-still -o img001.jpg`). ⚠️ Small flame (candle), outside or on a safe surface, water ready.
2. Upload to https://roboflow.com (free) → label boxes with class **`fire`** (and `smoke` if you want).
3. Export → format **YOLOv8** (or YOLO11) → "show download code" → paste your link in `ROBOFLOW_ZIP_URL` below.

ใช้ภาพจากกล้อง Pi ของเราเอง จะทำให้แม่นขึ้นมาก ใส่ภาพที่ "ไม่ใช่ไฟ" (ไฟสีส้ม, พระอาทิตย์ตก, ของสีแดง) โดยไม่ต้องตีกรอบ เพื่อลดการแจ้งเตือนผิด

In [ ]:
ROBOFLOW_ZIP_URL = ''   # paste Roboflow export link here, or leave empty to skip
if ROBOFLOW_ZIP_URL:
    !rm -rf /content/own && mkdir -p /content/own
    !curl -L "$ROBOFLOW_ZIP_URL" -o /content/own/own.zip && cd /content/own && unzip -q own.zip
    own_yaml = glob.glob('/content/own/**/data.yaml', recursive=True)[0]
    own_names = yaml.safe_load(open(own_yaml))['names']
    own_names = [own_names[k] for k in sorted(own_names)] if isinstance(own_names, dict) else list(own_names)
    print('your classes:', own_names)
    # Re-map your class ids to the D-Fire ids by name (e.g. your "fire"=0 -> D-Fire "fire"=1)
    remap = {i: names.index(n.lower()) for i, n in enumerate(own_names) if n.lower() in names}
    print('remap:', remap)
    for f in glob.glob('/content/own/**/labels/*.txt', recursive=True):
        out = []
        for line in open(f):
            p = line.split()
            if p and int(p[0]) in remap:
                out.append(' '.join([str(remap[int(p[0])])] + p[1:]))
        open(f, 'w').write('\n'.join(out) + ('\n' if out else ''))
    own_train = find_split('/content/own', ['train'])
    own_val = find_split('/content/own', ['valid', 'val'])
    data['train'] = [train_dir, own_train]
    if own_val:
        data['val'] = [val_dir, own_val]
    yaml.safe_dump(data, open(DATA_YAML, 'w'))
    print(open(DATA_YAML).read())

## 4. Train

- `yolo26n.pt` = nano model, the right size for a Pi 5.
- On a free T4, one epoch on D-Fire takes a few minutes. **50 epochs ≈ a few hours.** Start with `EPOCHS = 50`; use 100+ if you have time (Colab Pro / your own GPU).
- Training stops early by itself if it stops improving (`patience`).

In [ ]:
from ultralytics import YOLO
EPOCHS = 50
model = YOLO('yolo26n.pt')
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=640,
    batch=-1,          # auto batch size for the GPU
    patience=15,
    project=PROJECT,
    name='fire',
    exist_ok=True,
    cos_lr=True,
    plots=True,
)

In [ ]:
# ONLY if Colab disconnected during training: run setup cells (1), then this to continue.
# ใช้เฉพาะตอน Colab หลุดระหว่างเทรน: รันช่อง setup (1) ก่อน แล้วรันช่องนี้เพื่อเทรนต่อ
# from ultralytics import YOLO
# YOLO(f'{PROJECT}/fire/weights/last.pt').train(resume=True)

## 5. Check accuracy

- **mAP50** for class `fire`: higher is better. > 0.6 is usable for a robot, > 0.75 is good.
- Look at the confusion matrix and the test pictures. If orange lights are detected as fire → add "not fire" photos (step 3) and train again.

In [ ]:
BEST = f'{PROJECT}/fire/weights/best.pt'
best = YOLO(BEST)
m = best.val(data=DATA_YAML, imgsz=640, split='test' if 'test' in data else 'val')
for j, c in enumerate(m.box.ap_class_index):
    print(f'{best.names[int(c)]:>6}: mAP50 = {m.box.ap50[j]:.3f}   mAP50-95 = {m.box.ap[j]:.3f}')
print('all mAP50 =', round(m.box.map50, 3), ' mAP50-95 =', round(m.box.map, 3))
from IPython.display import Image, display
display(Image(f'{PROJECT}/fire/results.png', width=900))

In [ ]:
# Test at the SAME small size the Pi will use (320) to see the real quality on the robot.
# ทดสอบที่ขนาด 320 เท่ากับที่ใช้บน Pi
test_imgs = random.sample(glob.glob(f"{data.get('test', val_dir)}/*.jpg"), 6)
res = best.predict(test_imgs, imgsz=320, conf=0.5, verbose=False)
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for ax, r in zip(axs.flat, res):
    ax.imshow(r.plot()[:, :, ::-1]); ax.axis('off')
plt.show()

In [ ]:
# Test your own photo: upload a picture and see the result.
from google.colab import files
up = files.upload()
for f in up:
    r = best.predict(f, imgsz=320, conf=0.5, verbose=False)[0]
    plt.figure(figsize=(8, 6)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off'); plt.show()

## 6. Export for Raspberry Pi 5 (NCNN) and download

`imgsz=320` → fast on Pi 5. Server setting `YOLO_IMGSZ` must match (default 320).
If accuracy at 320 is too low for small/far fires, export with `imgsz=416` and set `YOLO_IMGSZ=416` on the Pi.

In [ ]:
EXPORT_SIZE = 320
path = best.export(format='ncnn', imgsz=EXPORT_SIZE)
print(path)
!rm -rf /content/fire_ncnn_model && cp -r "$path" /content/fire_ncnn_model
!cp "$BEST" /content/fire_best.pt
!cd /content && zip -qr fire_ncnn_model.zip fire_ncnn_model && ls -lh fire_ncnn_model.zip
files.download('/content/fire_ncnn_model.zip')
files.download('/content/fire_best.pt')   # keep the original too (to export other sizes later)

## 7. Put it on the Pi / นำไปใช้บน Pi

```bash
# copy fire_ncnn_model.zip to the Pi, then:
cd ~/server
unzip fire_ncnn_model.zip -d models/      # -> models/fire_ncnn_model/
pip install ultralytics ncnn              # in the server's venv (see README)
python3 app.py                            # log shows: [detector] YOLO model ... using=['fire']
```

Alert on smoke too: `FIRE_CLASSES=fire,smoke python3 app.py`